# Instruction-following: can the model copy what it is given?

**The gate.** `thought01.md` proposes teaching the model to *repeat* (`Say: {s_e}` / `Nói: {s_v}`) before
any mapping is trained. If the model already copies zero-shot, fine-tuning is unnecessary. If it does not,
the copy task is the first thing to fine-tune.

Number of sentences: **2,000** (deterministic sample of the MTET test split, seed 42).
Each model is prompted **in its own language**.

In [1]:
import csv, json, os, sys, warnings
warnings.filterwarnings("ignore")
os.environ.setdefault("TQDM_DISABLE", "1")
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name != "research-prototype" else Path.cwd()
sys.path.insert(0, str(ROOT))
import torch
from src import config as C, models as M, data as D

def read_csv(name):
    p = ROOT / "results" / name
    if not p.exists():
        print(f"missing {p.name} - run: python -m src.run_all (stage not executed yet)")
        return []
    return list(csv.DictReader(p.open(encoding="utf-8")))

print("python", sys.version.split()[0], "| torch", torch.__version__,
      "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")


python 3.12.10 | torch 2.7.1+cu128 | cuda True | NVIDIA GeForce RTX 3050 Laptop GPU


## 1. The exact prompt strings

In [2]:
from src import copy_eval

rows = copy_eval.sample_sentences(6)
for r in rows[:6]:
    print("vi:", r["vi"][:90])
    print("en:", r["en"][:90])
    print("   prompt(vi model):", C.MODELS["vit5-base"]["copy_prefix"] + r["vi"][:60])
    print("   prompt(en model):", C.MODELS["flan-t5-base"]["copy_prefix"] + r["en"][:60])
    print()

test split: 2536 pairs | diacritic swaps 0 | no-diacritics-both-sides 50
vi: Cái khát vọng của người phụ nữ có cho sự phát triển của bản thân là thứ quan trọng nhất .
en: The passion that the person has for her own growth is the most important thing .
   prompt(vi model): Nói: Cái khát vọng của người phụ nữ có cho sự phát triển của bản 
   prompt(en model): Say: The passion that the person has for her own growth is the mo

vi: Và đó là một bóng đèn nhỏ từ cây đèn pin vỡ .
en: And that 's a small torch from a broken flashlight .
   prompt(vi model): Nói: Và đó là một bóng đèn nhỏ từ cây đèn pin vỡ .
   prompt(en model): Say: And that 's a small torch from a broken flashlight .

vi: Tuy nhiên , từng chút từng chút một , sự hưng phấn từ các cuộc bầu cử và của cuộc cách mạn
en: However , bit by bit , the euphoria of the elections , and of the revolution as a whole , 
   prompt(vi model): Nói: Tuy nhiên , từng chút từng chút một , sự hưng phấn từ các cu
   prompt(en model): Say: However , b

## 2. Live sample — 8 prompts per base model

Only 8, so the cell stays fast. Section 3 shows the full 2,000-row result.

In [3]:
def demo(key, n=8):
    spec = C.MODELS[key]
    lang = spec["lang"]
    tok, model, _ = M.load(key)
    rs = copy_eval.sample_sentences(n)
    prompts = [spec["copy_prefix"] + r[lang] for r in rs]
    hyps = copy_eval._generate(tok, model, prompts, 96)
    for src, hyp in zip([r[lang] for r in rs], hyps):
        ok = "OK " if hyp.strip() == src.strip() else "BAD"
        print(f"{ok} src: {src[:80]}")
        print(f"    hyp: {hyp[:80]}")
    M.free(model)

demo("vit5-base")
demo("flan-t5-base")

loaded vit5-base        226.0M params  d_model=768 enc/dec=12/12  device=cuda dtype=bfloat16 vram=448 MB  3.87s
test split: 2536 pairs | diacritic swaps 0 | no-diacritics-both-sides 50


BAD src: Cái khát vọng của người phụ nữ có cho sự phát triển của bản thân là thứ quan trọ
    hyp: *+__Z_______________________________________________________________________Z___
BAD src: Và mảnh vườn , nó rất đẹp .
    hyp: *+_ đẹp ....Z ....[] mảnh vườn, nó rất đẹp ....! Và mảnh vườn, nó rất đẹp ....>.
BAD src: Và đó là một bóng đèn nhỏ từ cây đèn pin vỡ .
    hyp: * Và+ ._ RồiZ Và[ Rồi] Và!!> Rồi Rồij Và& VàW VàỴ VàẼ VàỠ VàẪ VàẲ VàẰ VàÕ VàẺ Và
BAD src: Tuy nhiên , từng chút từng chút một , sự hưng phấn từ các cuộc bầu cử và của cuộ
    hyp: *-+-_ --Z-----------------------------------------------------------------------
BAD src: Tôi im miệng , và lắng nghe họ . "
    hyp: * . "+ . " "_ . " "Z . " "[ . "] . "! . " "> " "j . " "& " " " Tôi lắng nghe họ 
BAD src: Đó sẽ là các danh nhân , và họ đang thực hiện điều đó ngay hiện nay
    hyp: * là các danh nhân, và họ đang thực hiện điều đó ngay bây giờ. Nói: Đó là các da
BAD src: Mỗi bức ảnh gợi nhắc về ai đó hoặc một cái gì đó , một nơi

loaded flan-t5-base     247.6M params  d_model=768 enc/dec=12/12  device=cuda dtype=bfloat16 vram=492 MB  3.13s
test split: 2536 pairs | diacritic swaps 0 | no-diacritics-both-sides 50


BAD src: The passion that the person has for her own growth is the most important thing .
    hyp: She is a person who is passionate about her own growth.
BAD src: And the garden , it was beautiful .
    hyp: It was a beautiful garden .
BAD src: And that 's a small torch from a broken flashlight .
    hyp: it 's a small flashlight .
BAD src: However , bit by bit , the euphoria of the elections , and of the revolution as 
    hyp: The euphoria of the elections was fading out .
BAD src: I shut up , and listen to them . "
    hyp: I 'm a snob .
BAD src: It will be entrepreneurs , and they 're doing it now .
    hyp: entrepreneurs are the ones who will be the ones who will be the ones who will be
BAD src: A photo is a reminder of someone or something , a place , a relationship , a lov
    hyp: it is a reminder of someone .
BAD src: How can you help ? You can become active , make noise . You can write a letter t
    hyp: You can volunteer to help spread the word .


## 3. Full result, 2,000 sentences x 4 models

In [4]:
table = read_csv("copy_summary.csv")
cols = ["model", "size_class", "n", "exact_match", "exact_match_norm", "bleu", "chrf", "tok_recall", "mean_hyp_words", "mean_ref_words"]
if table:
    print(" | ".join(cols))
    for r in table:
        print(" | ".join(str(r[c]) for c in cols))

model | size_class | n | exact_match | exact_match_norm | bleu | chrf | tok_recall | mean_hyp_words | mean_ref_words
vit5-base | base | 2000 | 0.0 | 0.0 | 13.25 | 19.04 | 0.2664 | 40.57 | 26.6
flan-t5-base | base | 2000 | 0.0265 | 0.029 | 8.4 | 23.44 | 0.3358 | 8.43 | 21.09
vit5-large | large | 2000 | 0.0 | 0.0 | 0.0 | 1.38 | 0.038 | 3.59 | 26.57


## 4. Failure taxonomy

Four failure shapes matter, because they need different fixes: **empty**, **truncated**, **looped**, **paraphrased**.

In [5]:
def classify(src, hyp):
    if not hyp.strip():
        return "empty"
    if hyp.strip() == src.strip():
        return "exact"
    if len(hyp.split()) < 0.8 * len(src.split()):
        return "truncated"
    if len(hyp.split()) > 1.8 * len(src.split()):
        return "looped"
    return "paraphrased"

for r in read_csv("copy_summary.csv"):
    key = r["model"]
    rows = read_csv(f"copy_{key}.csv")
    if not rows:
        continue
    from collections import Counter
    cnt = Counter(classify(x["src"], x["hyp"]) for x in rows)
    total = sum(cnt.values())
    print(key, {k: f"{v/total:.1%}" for k, v in cnt.most_common()})

vit5-base {'looped': '47.2%', 'truncated': '31.5%', 'paraphrased': '21.2%'}
flan-t5-base {'truncated': '79.8%', 'paraphrased': '16.7%', 'exact': '2.6%', 'looped': '0.8%'}
vit5-large {'truncated': '66.8%', 'empty': '30.9%', 'paraphrased': '2.2%'}


## 5. Verdict

In [6]:
for r in read_csv("copy_summary.csv"):
    verdict = "copies" if float(r["exact_match_norm"]) > 0.5 else "does not copy"
    print(f"{r['model']:<14} {verdict:<12} exact {float(r['exact_match_norm']):.1%}  BLEU {r['bleu']}")

vit5-base      does not copy exact 0.0%  BLEU 13.25
flan-t5-base   does not copy exact 2.9%  BLEU 8.4
vit5-large     does not copy exact 0.0%  BLEU 0.0


**What to conclude.** `exact_match_norm` is the number that decides whether fine-tuning on the copy task
is needed. BLEU on the copy task is a softer version of the same signal — it rewards partial copying that
exact match punishes, which is why both are reported.

If a model copies its own language reliably, the next experiment is the *mapping* (`small-experiment`),
not more fine-tuning.